# Week 6 (starter): Advanced Retrieval, Transform and Measure

Extends Week 5. The transformation shown here is pseudo-relevance feedback (PRF), which is fully local and needs no key; HyDE is guarded behind a key. Cells marked **TODO (you)** are yours. Dependency: `sentence-transformers`. For HyDE: `pip install openai` and a Gemini key.

In [ ]:
STOP=set('a an the of to in on for and or is are how do does what so that with by as it why'.split())
def retrieve_idx(q,k=K):
    qe=embedder.encode([q],convert_to_tensor=True,normalize_embeddings=True)
    sims=util.cos_sim(qe,doc_emb).cpu().numpy()[0]; return list(np.argsort(sims)[::-1][:k])
def prf_expand(q, feedback_k=3, add=4):
    top=retrieve_idx(q,feedback_k); words=re.findall(r'[a-zA-Z]+',' '.join(docs[i] for i in top).lower())
    cand=[w for w in words if w not in STOP and len(w)>3 and w not in q.lower()]
    return q+' '+' '.join(w for w,_ in Counter(cand).most_common(add))
def precision(idxs,topic): return sum(1 for i in idxs if topics[i]==topic)/K

print('per-query precision@4, baseline -> PRF:')
improved=regressed=0
for topic,q in queries:
    b=precision(retrieve_idx(q),topic); a=precision(retrieve_idx(prf_expand(q)),topic)
    flag='IMPROVED' if a>b else ('REGRESSED' if a<b else 'same')
    improved+=a>b; regressed+=a<b
    print(f'  [{topic:5s}] {b:.2f} -> {a:.2f}  {flag}   {q!r}')
print(f'\nimproved {improved}, regressed {regressed}  (a regression is your required failure case)')
# TODO (you): also add a re-ranking step, and explain the regression.

## HyDE (needs a key) and submit
HyDE generates a hypothetical answer and retrieves on it. With a key it slots in where PRF is; without one it is skipped.

In [ ]:
def hyde(q, model='gemini-2.5-flash-lite'):
    if not os.environ.get('GEMINI_API_KEY'):
        return '[API-BLOCKED] set GEMINI_API_KEY to generate a hypothetical answer, then embed and retrieve on it'
    from openai import OpenAI
    c=OpenAI(api_key=os.environ['GEMINI_API_KEY'], base_url='https://generativelanguage.googleapis.com/v1beta/openai/')
    return c.chat.completions.create(model=model, messages=[{'role':'user','content':'Write a one-sentence answer to: '+q}]).choices[0].message.content
print(hyde('how do indexes speed up reads'))
# Submit: PR with before/after deltas + a concrete cross-repo artifact with a classmate.
# Rubric: transform+rerank (30), before/after deltas (30), regression explained (20), cross-repo artifact (20).

## Part 1: Transform and re-rank
Extend your Week 5 retriever with at least one query transformation and a re-ranking step over the retrieved candidates.

In [1]:
import os
import re
import pathlib
import numpy as np
import pandas as pd

from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
import faiss

os.environ["HF_HOME"] = str(
    (pathlib.Path(".") / ".hf_cache").resolve()
)

embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

c:\Users\wyatt\Documents\Gradschool\cosc_650\repos\cosc-650-applied-llm-systems\src\week6\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4293.81it/s]


In [2]:
DOCS_DIR = pathlib.Path("../week5/gardening_docs")

def load_pdf(file_path):
    reader = PdfReader(file_path)
    pages = []

    for page in reader.pages:
        text = page.extract_text()

        if text:
            pages.append(text)

    return "\n\n".join(pages)


documents = []

for file_path in DOCS_DIR.glob("*.pdf"):
    documents.append({
        "source": file_path.name,
        "text": load_pdf(file_path)
    })


print(f"Loaded {len(documents)} documents")

for doc in documents:
    print(f"{doc['source']}: {len(doc['text']):,} characters")

Loaded 8 documents
bush_gardening.pdf: 147,544 characters
fruit_gardening.pdf: 24,727 characters
gardening_beds.pdf: 20,128 characters
gardening_pests.pdf: 10,832 characters
gardening_planters.pdf: 34,943 characters
grass_gardening.pdf: 33,280 characters
tree_planting.pdf: 23,515 characters
vegetable_gardening.pdf: 78,865 characters


In [3]:
def chunk_paragraph(text):
    return [re.sub(r'\s+',' ',p).strip() for p in text.split('\n\n') if p.strip()]

def chunk_documents(documents, chunker):
    chunks = []

    for doc in documents:
        doc_chunks = chunker(doc["text"])

        for chunk_id, chunk in enumerate(doc_chunks):
            chunks.append({
                "source": doc["source"],
                "chunk_id": chunk_id,
                "text": chunk
            })

    return chunks


configs = {
    "by-paragraph": chunk_documents(
        documents,
        chunk_paragraph
    )
}

for n, c in configs.items():
    print(f'{n:16s} -> {len(c)} chunks')

def build(chunks):
    texts = [chunk["text"] for chunk in chunks]

    e = embedder.encode(
        texts,
        normalize_embeddings=True
    ).astype("float32")

    idx = faiss.IndexFlatIP(e.shape[1])
    idx.add(e)

    return idx

def retrieve(idx, chunks, q, k=3):
    qe = embedder.encode(
        [q],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = idx.search(qe, k)

    results = []

    for score, i in zip(scores[0], indices[0]):
        if i >= 0:
            results.append({
                "source": chunks[i]["source"],
                "chunk_id": chunks[i]["chunk_id"],
                "text": chunks[i]["text"],
                "score": float(score)
            })

    return results

by-paragraph     -> 200 chunks


In [4]:
indices = {
    name: build(chunks)
    for name, chunks in configs.items()
}

for name, idx in indices.items():
    print(f"{name:16s} -> {idx.ntotal} vectors")

by-paragraph     -> 200 vectors


In [ ]:
test_queries = [
    "How much sunlight do vegetables need?",
    "How should I choose a location for a vegetable garden?",
    "What type of soil is best for planting fruit trees?",
    "How should fruit trees be watered after planting?",
    "What are the benefits of using raised garden beds?",
    "How wide should a raised garden bed be?",
    "How can crop rotation help control garden insects?",
    "How should shrubs be selected for different sunlight conditions?",
    "What should I consider when selecting a tree planting site?",
    "How should a lawn be watered and maintained?"
]

# Week 5 baseline metrics for by-paragrpah chunking
baseline_metrics = {
    "precision@3": 0.433,
    "recall@3": 0.779,
    "hit@3": 0.900
}

## Part 2: Measure the effect
Run the same query set as Week 5 and report retrieval quality before and after, with per-query deltas. State which query types benefited and which did not.

## Part 3: Find one failure and explain it
Show at least one query where the transformation made retrieval worse, and explain why the technique misfired, since a transformation is an assumption about the query and assumptions break. If no query regressed, report that result and explain why the transformation held across your set.